Dataset Downloaded from: https://huggingface.co/datasets/CiferAI/Cifer-Fraud-Detection-Dataset-AF

## DATASET DOWNLOAD, LOADING AND PARTITIONING

In [1]:
# !pip install pyspark
from datasets import load_dataset
import os
import pyarrow.parquet as pq
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, countDistinct

#### Downloading the Dataset

In [2]:
# Loading the fraud detection dataset from Hugging Face
fraud_detection_dataset = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-1-(…):   0%|          | 0.00/138M [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-10(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-11(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

...

Cifer-Fraud-Detection-Dataset-AF-part-6-(…):   0%|          | 0.00/131M [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-7-(…):   0%|          | 0.00/131M [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-8-(…):   0%|          | 0.00/131M [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-9-(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21000000 [00:00<?, ? examples/s]

In [3]:
fraud_training_records = fraud_detection_dataset["train"]

#### Saving Dataset Into Multiple Paraquet

In [4]:
# Creating directory to store partitioned parquet files
output_directory = "data/raw/cifer-fraud-detection-AF/train"
os.makedirs(output_directory, exist_ok=True)

In [5]:
rows_per_partition = 1_000_000

In [6]:
total_rows = fraud_training_records.num_rows

for partition_number, start_index in enumerate(range(0, total_rows, rows_per_partition), start=1):

    end_index = min(start_index + rows_per_partition, total_rows)

    arrow_partition = fraud_training_records.select(
        range(start_index, end_index)
    ).data.table

    pq.write_table(
        arrow_partition,
        os.path.join(
            output_directory,
            f"train_part_{partition_number}.parquet"
        )
    )

    print(f"Partition {partition_number} saved.")

Partition 1 saved.
Partition 2 saved.
Partition 3 saved.
Partition 4 saved.
Partition 5 saved.
Partition 6 saved.
Partition 7 saved.
Partition 8 saved.
Partition 9 saved.
Partition 10 saved.
Partition 11 saved.
Partition 12 saved.
Partition 13 saved.
Partition 14 saved.
Partition 15 saved.
Partition 16 saved.
Partition 17 saved.
Partition 18 saved.
Partition 19 saved.
Partition 20 saved.
Partition 21 saved.


#### Creating the PySpark Session



In [7]:
spark_session = (
    SparkSession.builder
    .appName("Task1_session")
    .config("spark.driver.memory", "2g")   # limit memory
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "50")  # reduce load
    .getOrCreate()
)

#### Loading Dataset into PySpark and Initial Inspections

In [8]:
fraud_detection_dataframe = spark_session.read.parquet(output_directory)

In [9]:
print("=" * 80)
print("FIRST 20 RECORDS")
print("=" * 80)
fraud_detection_dataframe.show(20, truncate=False)

FIRST 20 RECORDS
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|type    |amount   |nameOrig   |oldbalanceOrg|newbalanceOrig|nameDest   |oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|143 |PAYMENT |728.68   |C895984064 |157370.38    |296733.47     |M1248796614|946814.15     |21827.99      |0      |0             |
|83  |PAYMENT |37025.37 |C1166835933|12.44        |23912.77      |C266527134 |435103.83     |18206.87      |0      |0             |
|103 |CASH_IN |331069.91|C1589363125|82.16        |59517.13      |C222506215 |1383838.87    |1740585.08    |0      |0             |
|77  |PAYMENT |5097.55  |C109893045 |1272721.12   |1954746.32    |C250174919 |565635.26     |40351.13      |0      |0             |
|53  |CASH_IN |240031.01|C168215249 |1514711.38   |1464170.

In [10]:
print("=" * 80)
print("DATASET SCHEMA")
print("=" * 80)
fraud_detection_dataframe.printSchema()

DATASET SCHEMA
root
 |-- step: long (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: long (nullable = true)
 |-- isFlaggedFraud: long (nullable = true)



In [11]:
total_record_count = fraud_detection_dataframe.count()
total_column_count = len(fraud_detection_dataframe.columns)

print(f"Total Records : {total_record_count:,}")
print(f"Total Columns : {total_column_count}")

print("\nColumn Names")

for column_name in fraud_detection_dataframe.columns:
    print(f"- {column_name}")

Total Records : 21,000,000
Total Columns : 11

Column Names
- step
- type
- amount
- nameOrig
- oldbalanceOrg
- newbalanceOrig
- nameDest
- oldbalanceDest
- newbalanceDest
- isFraud
- isFlaggedFraud


The totoal instance count for the dataset is 21 million records (> 10 million) with 11 columns (>10 columns). Thus, it satistifies the requirement of the assignment.

#### Checking File Size of Paraquet

In [12]:
print("\n" + "=" * 80)
print("PARQUET FILES CREATED")
print("=" * 80)
output_directory = "data/raw/cifer-fraud-detection-AF/train"
parquet_file_count = len([
    file_name
    for file_name in os.listdir(output_directory)
    if file_name.endswith(".parquet")
])

print(f"Number of Parquet Files : {parquet_file_count}")

print("\nNotebook verification completed successfully.")


PARQUET FILES CREATED
Number of Parquet Files : 21

Notebook verification completed successfully.


In [13]:
print("PARQUET FILE SIZE")
print("=" * 80)

total_dataset_size_bytes = 0

for current_directory, _, parquet_files in os.walk(output_directory):

    for parquet_file in parquet_files:

        parquet_file_path = os.path.join(
            current_directory,
            parquet_file
        )

        total_dataset_size_bytes += os.path.getsize(
            parquet_file_path
        )

dataset_size_megabytes = total_dataset_size_bytes / (1024 ** 2)
dataset_size_gigabytes = total_dataset_size_bytes / (1024 ** 3)

print(f"Dataset Size : {dataset_size_megabytes:.2f} MB")
print(f"Dataset Size : {dataset_size_gigabytes:.2f} GB")

PARQUET FILE SIZE
Dataset Size : 1061.43 MB
Dataset Size : 1.04 GB


#### Summary Statistics

In [15]:
fraud_detection_dataframe.describe().show()

+-------+------------------+--------+------------------+-------------+------------------+------------------+-------------+------------------+--------------------+--------------------+--------------------+
|summary|              step|    type|            amount|     nameOrig|     oldbalanceOrg|    newbalanceOrig|     nameDest|    oldbalanceDest|      newbalanceDest|             isFraud|      isFlaggedFraud|
+-------+------------------+--------+------------------+-------------+------------------+------------------+-------------+------------------+--------------------+--------------------+--------------------+
|  count|          21000000|21000000|          21000000|     21000000|          21000000|          21000000|     21000000|          21000000|            21000000|            21000000|            21000000|
|   mean|238.86997738095238|    NULL|179899.52085984533|         NULL|1337090.1914971431|2089204.3027557558|         NULL|1939226.6123958612|  3756326.4135460616|0.001308095238095.

In [16]:
print("COLUMN DATA TYPES")
print("=" * 80)

for column_name, data_type in fraud_detection_dataframe.dtypes:
    print(f"{column_name:<35}{data_type}")

COLUMN DATA TYPES
step                               bigint
type                               string
amount                             double
nameOrig                           string
oldbalanceOrg                      double
newbalanceOrig                     double
nameDest                           string
oldbalanceDest                     double
newbalanceDest                     double
isFraud                            bigint
isFlaggedFraud                     bigint
